In [1]:
import time
import multiprocessing
import numpy as np


def monte_carlo_worker(num_samples: int) -> int:
    """Menghitung jumlah titik acak yang jatuh di dalam lingkaran satuan."""
    rng = np.random.default_rng()  # Seed PRNG independen tiap worker
    x = rng.uniform(-1.0, 1.0, num_samples)
    y = rng.uniform(-1.0, 1.0, num_samples)
    return int(np.sum(x * x + y * y <= 1.0))


def main():
    TOTAL_SAMPLES = 20_000_000
    # Google Colab standar biasanya menyediakan 2 core CPU (atau sesuaikan dengan os.cpu_count())
    num_workers = 2

    print(f"Memulai Simulasi Monte Carlo dengan N = {TOTAL_SAMPLES:,} sampel...\n")

    # 1. Eksekusi Sekuensial (1 Core)
    t0 = time.perf_counter()
    inside_seq = monte_carlo_worker(TOTAL_SAMPLES)
    pi_seq = 4.0 * inside_seq / TOTAL_SAMPLES
    time_seq = time.perf_counter() - t0

    # 2. Eksekusi Paralel (Multi-Core)
    samples_per_worker = TOTAL_SAMPLES // num_workers
    t1 = time.perf_counter()
    with multiprocessing.Pool(processes=num_workers) as pool:
        counts = pool.map(monte_carlo_worker, [samples_per_worker] * num_workers)
    pi_par = 4.0 * sum(counts) / (samples_per_worker * num_workers)
    time_par = time.perf_counter() - t1

    speedup = time_seq / time_par
    efisiensi = (speedup / num_workers) * 100

    print("=== HASIL PRAKTIKUM KOMPUTASI PARALEL (MONTE CARLO) ===")
    print(f"Sekuensial (1 Core)        : {time_seq:.4f}s | Estimasi Pi = {pi_seq:.6f}")
    print(f"Paralel ({num_workers} Workers)       : {time_par:.4f}s | Estimasi Pi = {pi_par:.6f}")
    print(f"Speedup                    : {speedup:.2f}x")
    print(f"Efisiensi Paralel          : {efisiensi:.1f}%")


if __name__ == "__main__":
    main()

Memulai Simulasi Monte Carlo dengan N = 20,000,000 sampel...

=== HASIL PRAKTIKUM KOMPUTASI PARALEL (MONTE CARLO) ===
Sekuensial (1 Core)        : 0.6058s | Estimasi Pi = 3.142003
Paralel (2 Workers)       : 0.5244s | Estimasi Pi = 3.141451
Speedup                    : 1.16x
Efisiensi Paralel          : 57.8%
